In [5]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv 
load_dotenv()
llm= ChatOpenAI(model="gpt-5.4-nano-2026-03-17",temperature=0)

In [6]:
from langchain.tools import tool

In [7]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response

In [8]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [9]:

@tool 
def tool_wikipedia_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about persons, places, etc. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response
    

In [10]:
tool_duckduckgo_search.invoke("What is the capital of France?")

'Paris[a] is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km 2 (40.7 sq mi), and a metropolitan population of 13.2 million as of January 2026. [4] Located on the river Seine in the centre of the Île-de-France region, it is the largest metropolitan area and fourth-most populous city in the European Union (EU). Nicknamed the "City ... Learn why Paris is the capital of France and how it became a global city with a rich cultural heritage. Discover its geography, climate, population, landmarks, and industries. Paris is the capital city of France, situated on the River Seine, in northern France, at the heart of the Île-de-France region ("Région parisienne"). Jun 29, 2018 · Paris is the capital city of France, as well as the country\'s largest city. Jun 2, 2019 · Paris, the capital city of France, is one of the most famous and influential cities in the world. Located in the northern part of the country along the Seine River, Par

In [11]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [12]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [13]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

### Bind Tools

In [14]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info
]

In [15]:
llm_bind = llm.bind_tools(toolkit)

In [16]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 282, 'total_tokens': 302, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DXNuKOyFSk66tavjdScfbqzBrdAXw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db475-8b1e-7ed2-85ee-fd5ad7d2c903-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_KWklRq3FQp8OzQiMpwT9krXb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 282, 'output_tokens': 20, 'total_tokens': 302, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### ReactAgent

In [17]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [18]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='a81ed888-f283-4cc2-9afb-40d64f1ccfc7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 282, 'total_tokens': 302, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DXO0oHiqbHjcSWU8yHKG3yXeFkzbv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db47b-ac26-73d2-b148-601c29e11779-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_XbwzEv9d7v35Ximw2EI0s2ZS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada